# ASCII Maze RL Workshop — Mac (MLX)

Train a small LLM to solve ASCII mazes using GRPO (Group Relative Policy Optimization).
Understand how rollouts and the reward function interact to enable learning.

**Your task:** Design a reward function that teaches the model to solve mazes.

The model has already been fine-tuned (SFT) to understand the maze format and output move sequences.
Your reward function will guide GRPO to improve its maze-solving accuracy.

## Pipeline
1. **SFT** (done for you) → model knows the format, solves 100% of 3×3, 88% of 4×4, 32% of 5×5
2. **GRPO** (your exercise) → reward function refines accuracy on 4×4 and 5×5

## Requirements

- Mac with Apple Silicon (M1+) and ≥16GB unified memory
- Python 3.12+, [uv](https://docs.astral.sh/uv/) installed

## Setup

From the repo root:

```bash
uv sync                # install all dependencies
```

**Cursor / VS Code:** Open this notebook and select the `.venv (Python 3.x)` kernel
when prompted. If asked to install `ipykernel`, click Install.

**Jupyter (standalone):**
```bash
uv run jupyter notebook notebooks/maze_grpo_mac.ipynb
```

This notebook uses **MLX** — Apple's framework for efficient ML on Apple Silicon.

In [ ]:
import sys, os
# Ensure we can import from the repo root
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
if os.path.exists('src') and '' not in sys.path:
    sys.path.insert(0, '')
elif os.path.exists('../src'):
    os.chdir('..')
    sys.path.insert(0, '')

import mlx.core as mx
print(f"MLX backend: {mx.default_device()}")
print(f"Python: {sys.version}")

## 1. Download the Pre-trained SFT Adapter

The SFT model is a LoRA adapter trained on Mac with MLX, then converted to
PEFT format and published to HuggingFace. We download it and convert back
to MLX LoRA format so `train_grpo.py` can load it.

The base model is `mlx-community/Qwen2.5-0.5B-Instruct-4bit` — a 4-bit
quantized model that fits comfortably in Apple Silicon unified memory.

In [ ]:
from pathlib import Path
import subprocess

BASE_MODEL = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
SFT_ADAPTER_HF = "StephenJHardy/maze-sft-qwen2.5-0.5b"
SFT_ADAPTER_DIR = Path("checkpoints/sft-adapter")

# Download the base model (cached by mlx_lm after first run)
from mlx_lm import load
print(f"Loading base model: {BASE_MODEL}")
_model, _tokenizer = load(BASE_MODEL)
del _model
print("Base model cached.")

# Download and convert the PEFT adapter to MLX format
if not (SFT_ADAPTER_DIR / "adapters.safetensors").exists():
    SFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    print(f"\nDownloading SFT adapter: {SFT_ADAPTER_HF}")
    from huggingface_hub import hf_hub_download
    from safetensors.numpy import load_file, save_file

    peft_path = hf_hub_download(SFT_ADAPTER_HF, "adapter_model.safetensors")
    peft_weights = load_file(peft_path)

    # Convert PEFT → MLX: transpose weights, fix key names
    mlx_weights = {}
    for key, value in peft_weights.items():
        mlx_key = key.replace("base_model.model.model.layers.", "model.layers.")
        mlx_key = mlx_key.replace(".lora_A.weight", ".lora_a")
        mlx_key = mlx_key.replace(".lora_B.weight", ".lora_b")
        mlx_weights[mlx_key] = value.T

    save_file(mlx_weights, str(SFT_ADAPTER_DIR / "adapters.safetensors"))
    print(f"Converted {len(peft_weights)} PEFT weights → MLX format")
    print(f"Saved to {SFT_ADAPTER_DIR}")
else:
    print(f"SFT adapter already at {SFT_ADAPTER_DIR}")

## 2. Explore the Maze Format

Let's see what mazes look like and how solutions work.

In [ ]:
from src.maze_gen import generate
from src.maze_repr import to_str, to_prompt, SYSTEM_PROMPT, solution_to_str
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress
from src.reward import compute_reward

maze = generate(4, 4, seed=42)
print("Maze (4×4):")
print(to_str(maze))
print(f"\nSolution: {solution_to_str(maze.solution_moves)}")
print(f"Solution length: {len(maze.solution_moves)} moves")
print(f"\nEntry: {maze.entry}, Exit: {maze.exit}")

In [ ]:
print("System prompt:")
print(SYSTEM_PROMPT)
print("\nUser message (just the maze grid):")
print(to_str(maze))

In [ ]:
test_output = "r r d d d r"
moves = extract_moves(test_output)
print(f"Parsed moves: {moves}")

path = simulate(moves, maze)
print(f"Path through maze: {path}")
print(f"Reached exit: {reached_exit(path, maze)}")
print(f"Valid steps: {len(path) - 1} / {len(moves)}")
print(f"Progress toward exit: {manhattan_progress(path[-1], maze.exit, maze.entry):.2f}")

## 3. Load Model & Check Baseline

The pre-trained SFT model (4-bit quantized, 5000 iterations on Mac) solves:
- 3×3: 100%
- 4×4: 88%
- 5×5: 32%
- 6×6: 13%
- 7×7: 5%

Your reward function will guide GRPO to improve 4×4 and 5×5.

In [ ]:
from mlx_lm import load
import mlx_lm
from mlx_lm.sample_utils import make_sampler
from src.train_grpo import setup_model

# Load base model + SFT adapter for a quick demo
model, tokenizer = load(BASE_MODEL)

# Load SFT adapter weights
import mlx.core as mx
adapter_weights = mx.load(str(SFT_ADAPTER_DIR / "adapters.safetensors"))
from mlx_lm.tuner.utils import linear_to_lora_layers
from src.train_grpo import find_layers

num_layers = len(find_layers(model))
linear_to_lora_layers(model, num_layers=num_layers, config={
    "rank": 16, "alpha": 16.0, "dropout": 0.0, "scale": 1.0,
    "keys": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj"],
})
model.load_weights(list(adapter_weights.items()), strict=False)
model.eval()
print(f"Loaded: {BASE_MODEL} + SFT adapter")

# Quick test: generate a solution for our sample maze
prompt = to_prompt(maze, tokenizer=tokenizer)
sampler = make_sampler(temp=0.0)
response = mlx_lm.generate(model, tokenizer, prompt=prompt, max_tokens=30, sampler=sampler)
print(f"\nModel output: {response!r}")
test_moves = extract_moves(response)
if test_moves:
    path = simulate(test_moves, maze)
    print(f"Solved: {reached_exit(path, maze)}")
del model

### 3b. Evaluate SFT Baseline

Run this now to get a reference point before GRPO training.

In [ ]:
from src.evaluate import load_model_for_eval, evaluate_dataset, print_summary
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig
from src.make_eval_splits import EVAL_SMALL

# Generate eval data
Path('data').mkdir(exist_ok=True)
eval_ds = MazeDataset.generate(EVAL_SMALL, progress=False)
print(f"Eval data: {eval_ds.summary()}")

In [ ]:
# Evaluate baseline (SFT only, no GRPO adapters)
print("=" * 60)
print("BASELINE (SFT only)")
print("=" * 60)
base_model, base_tok = load_model_for_eval(BASE_MODEL, adapter_path=str(SFT_ADAPTER_DIR))
base_results, base_summary = evaluate_dataset(
    base_model, base_tok, eval_ds,
    max_tokens=40, temperature=0.0, verbose=True,
)
print_summary(base_summary)
del base_model

## 4. Design Your Reward Function ⭐

**This is the main exercise.** Design a reward function that gives
GRPO useful signal to learn from.

We've pre-generated rollouts (8 attempts at each maze, at multiple
temperatures). You can re-score them instantly with different reward
functions — no model inference needed for this exploration phase.

### The challenge
GRPO compares rollouts within a group. If all rollouts score the same,
advantages are zero and GRPO is blind. Your reward must create **variance**
between rollouts while being **informative** (better paths score higher).

### Available tools
- `extract_moves(text)` → parse output to list of moves, or None
- `simulate(moves, maze)` → walk through maze, returns path
- `reached_exit(path, maze)` → did it solve?
- `manhattan_progress(pos, target, origin)` → distance progress (0–1)
- `solve(width, height, walls, start, end)` → BFS shortest path

In [ ]:
from src.maze_gen import Maze, solve as bfs_solve
from src.maze_verify import extract_moves, simulate, reached_exit, manhattan_progress


def reconstruct_maze(walls, entry, exit_, width, height, solution_path):
    frozen_walls = frozenset(frozenset(tuple(c) for c in w) for w in walls)
    return Maze(width=width, height=height, walls=frozen_walls,
               entry=tuple(entry), exit=tuple(exit_),
               solution=tuple(tuple(p) for p in solution_path), seed=0)


def maze_reward(completion, maze):
    """
    Your reward function. Scores a single completion against a maze.
    Start simple (v1), then iterate based on the analysis below.

    Args:
        completion: model output string (e.g. "r r d d")
        maze: Maze object

    Returns:
        float reward score
    """
    # --- v1: naive binary reward (start here) ---
    moves = extract_moves(completion)
    if moves is None:
        return 0.0
    path = simulate(moves, maze)
    return 1.0 if reached_exit(path, maze) else 0.0

### 4b. Test Your Reward on Pre-generated Rollouts

Re-run this cell after changing your reward function above.
Scoring is instant — no GPU/model inference needed.

In [ ]:
import json as _json
import numpy as np

with open('configs/pregenerated_rollouts.json') as f:
    rollout_data = _json.load(f)

mazes_meta = rollout_data["mazes"]
all_completions = rollout_data["completions"]
temperatures = rollout_data["temperatures"]

print(f"{'Temp':>4s} | {'Mean Reward':>11s} | {'Mean Std':>9s} | "
      f"{'Solved':>8s} | {'Zero-var mazes':>14s}")
print("-" * 60)

for temp in temperatures:
    temp_key = str(temp)
    all_stds = []
    all_rewards = []
    total_solved = 0
    zero_var = 0
    total = len(mazes_meta) * 8

    for maze_idx, meta in enumerate(mazes_meta):
        comps = all_completions[temp_key][maze_idx]
        maze_obj = reconstruct_maze(
            meta["walls"], meta["entry"], meta["exit"],
            meta["width"], meta["height"], meta["solution_path"],
        )
        rewards = [maze_reward(c, maze_obj) for c in comps]
        std = np.std(rewards)
        all_stds.append(std)
        all_rewards.append(np.mean(rewards))
        if std < 0.005:
            zero_var += 1
        for c in comps:
            moves = extract_moves(c)
            if moves and reached_exit(simulate(moves, maze_obj), maze_obj):
                total_solved += 1

    print(f"{temp:4.1f} | {np.mean(all_rewards):>11.4f} | {np.mean(all_stds):>9.4f} | "
          f"{total_solved:>3d}/{total:<3d} | {zero_var:>6d}/{len(mazes_meta):<6d}")

print("\nGood reward functions have:")
print("   - Mean Std > 0.15 at temp=1.0")
print("   - Few zero-variance mazes")
print("   - If ALL mazes are zero-var, GRPO learns nothing!")

In [ ]:
# Visualize rollouts at a chosen temperature with your reward function
TEMP = "1.0"  # try: "0.6", "0.8", "1.0", "1.2", "1.4", "1.6", "1.8"

from src.rollout_capture import MazeRollouts, RolloutResult, compute_advantages
from src.build_rollout_viewer import HTML_TEMPLATE
from dataclasses import asdict
from IPython.display import HTML, display

viz_rollouts = []
for maze_idx, meta in enumerate(mazes_meta):
    comps = all_completions[TEMP][maze_idx]
    maze_obj = reconstruct_maze(
        meta["walls"], meta["entry"], meta["exit"],
        meta["width"], meta["height"], meta["solution_path"],
    )
    rewards = [maze_reward(c, maze_obj) for c in comps]
    advantages, mean_r, std_r = compute_advantages(rewards)

    rollouts = []
    for j, c in enumerate(comps):
        moves = extract_moves(c)
        path = simulate(moves or [], maze_obj)
        rollouts.append(RolloutResult(
            completion=c, moves_parsed=moves,
            path=[list(p) for p in path], reward=rewards[j],
            solved=reached_exit(path, maze_obj),
            valid_steps=len(path)-1,
            progress=max(manhattan_progress(path[-1], maze_obj.exit, maze_obj.entry), 0.0),
        ))

    viz_rollouts.append(MazeRollouts(
        maze_id=meta["id"], width=meta["width"], height=meta["height"],
        maze_str=meta["maze_str"], entry=meta["entry"], exit=meta["exit"],
        correct_path=meta["solution_path"],
        correct_moves=meta["solution_moves"].split() if isinstance(meta["solution_moves"], str) else meta["solution_moves"],
        solution_length=meta["solution_length"],
        rollouts=rollouts, advantages=advantages,
        reward_mean=mean_r, reward_std=std_r,
    ))

import json as _json
viewer_data = [asdict(r) for r in viz_rollouts]
serialized = _json.dumps(viewer_data).replace('</', '<\\/')
viewer_html = HTML_TEMPLATE.replace('__DATA_PLACEHOLDER__', serialized)

with open('rollout_viewer.html', 'w') as f:
    f.write(viewer_html)
print(f'Rollout viewer (temp={TEMP}): {len(viz_rollouts)} mazes')
print(f'Saved to rollout_viewer.html — open in a browser for the full interactive viewer.')

display(HTML(f'<iframe srcdoc="{viewer_html.replace(chr(34), "&quot;")}" '
            f'width="100%" height="600px" style="border:1px solid #333;'
            f'border-radius:4px;"></iframe>'))

### 4c. Iterate

Go back to the reward function cell, improve it, re-run the scoring.

**Progression:**
1. Binary (0/1) → notice high zero-var count
2. Add `-1.0` for unparseable → differentiates gibberish from failures
3. Add partial credit (coverage/progress) → reduces zero-var
4. Use BFS distance or manhattan → more accurate progress
5. Add loop penalty → discourages random wandering

Once you're happy with the variance, move to GRPO training below.

## 5. Generate Training Data & Run GRPO

Lock in your reward function and train with the MLX GRPO loop.
~15-20 min on a Mac for 500 steps.

In [ ]:
from src.maze_dataset import MazeDataset, DatasetConfig, SizeConfig

train_config = DatasetConfig(
    name="grpo_exercise",
    algorithm="wilson",
    sizes=[
        SizeConfig(width=3, height=3, count=200, start_seed=500_000),
        SizeConfig(width=4, height=4, count=400, start_seed=510_000),
        SizeConfig(width=5, height=5, count=200, start_seed=520_000),
    ],
)
train_ds = MazeDataset.generate(train_config, progress=False)
print(f"Training data: {train_ds.summary()}")

### 5b. Write Your Reward into `src/reward.py`

The GRPO training loop in `src/train_grpo.py` calls `compute_reward(completion, maze)` from `src/reward.py`.
Copy your reward function there now.

The function signature is:
```python
def compute_reward(completion: str, maze: Maze) -> float:
```

You can either:
1. Edit `src/reward.py` directly in your editor, or
2. Overwrite it from this notebook (uncomment and edit the cell below)

In [ ]:
# Uncomment to overwrite src/reward.py with your reward function.
# Edit the body of compute_reward to match what you designed above.

# reward_code = '''
# from __future__ import annotations
# from src.maze_gen import Maze
# from src.maze_verify import extract_moves, manhattan_progress, reached_exit, simulate
#
# def compute_reward(completion: str, maze: Maze) -> float:
#     moves = extract_moves(completion)
#     if moves is None:
#         return -1.0
#     path = simulate(moves, maze)
#     if reached_exit(path, maze):
#         efficiency = (len(maze.solution) - 1) / max(len(moves), len(maze.solution) - 1)
#         return 0.6 + 0.4 * efficiency
#     valid_steps = len(path) - 1
#     optimal_len = len(maze.solution) - 1
#     coverage = min(valid_steps / max(optimal_len, 1), 1.0)
#     progress = max(manhattan_progress(path[-1], maze.exit, maze.entry), 0.0)
#     return 0.5 * (0.7 * coverage + 0.3 * progress)
#
# def reward_fn_for_maze(maze: Maze):
#     def reward_fn(response: str, ground_truth: str) -> float:
#         return compute_reward(response, maze)
#     return reward_fn
# '''.strip()
#
# with open('src/reward.py', 'w') as f:
#     f.write(reward_code + '\n')
# print("Updated src/reward.py")

# Reload the module so the training loop picks up your changes
import importlib
import src.reward
importlib.reload(src.reward)
from src.reward import compute_reward
print(f"Reward function loaded: {compute_reward.__module__}")

In [ ]:
# Save training data
from pathlib import Path
Path('data').mkdir(exist_ok=True)
train_ds.save('data/train_grpo_exercise.jsonl')
print(f"Saved {len(train_ds)} mazes to data/train_grpo_exercise.jsonl")

### 5c. Run GRPO Training

This calls the MLX GRPO loop from `src/train_grpo.py`.
It takes ~15-20 minutes for 500 steps on an M1/M2/M3 Mac.

Watch the metrics:
- `reward_mean` should increase over time
- `reward_max` hitting 1.0 means some rollouts solve the maze
- `reward_std > 0` means there's variance for GRPO to learn from

In [ ]:
import subprocess, sys

result = subprocess.run([
    sys.executable, "-m", "src.train_grpo",
    "--model", BASE_MODEL,
    "--adapters", str(SFT_ADAPTER_DIR),
    "--dataset", "data/train_grpo_exercise.jsonl",
    "--max-steps", "500",
    "--num-generations", "8",
    "--temperature", "1.0",
    "--lr", "1e-5",
    "--beta", "0.04",
    "--max-tokens", "30",
    "--log-interval", "10",
    "--save-interval", "100",
    "--output-dir", "checkpoints/grpo-exercise",
], text=True, capture_output=False)

print(f"\nExit code: {result.returncode}")

## 6. Evaluate After GRPO

Compare your GRPO-trained model against the baseline you measured earlier.

In [ ]:
# Evaluate after GRPO (pick the latest checkpoint)
import glob

ckpt_dirs = sorted(glob.glob('checkpoints/grpo-exercise/step-*'))
if ckpt_dirs:
    latest_ckpt = ckpt_dirs[-1]
    print("=" * 60)
    print(f"AFTER GRPO ({latest_ckpt})")
    print("=" * 60)
    grpo_model, grpo_tok = load_model_for_eval(BASE_MODEL, adapter_path=latest_ckpt)
    grpo_results, grpo_summary = evaluate_dataset(
        grpo_model, grpo_tok, eval_ds,
        max_tokens=40, temperature=0.0, verbose=True,
    )
    print_summary(grpo_summary)

    # Side-by-side comparison
    print("\nComparison:")
    print(f"{'Size':>5s}  {'Before':>8s}  {'After':>8s}  {'Change':>8s}")
    print("-" * 35)
    for size in sorted(set(list(base_summary.by_size.keys()) + list(grpo_summary.by_size.keys()))):
        pre_rate = base_summary.by_size.get(size, {}).get('solve_rate', 0) * 100
        post_rate = grpo_summary.by_size.get(size, {}).get('solve_rate', 0) * 100
        print(f"{size:>5s}  {pre_rate:7.1f}%  {post_rate:7.1f}%  {post_rate-pre_rate:+7.1f}pp")
    del grpo_model
else:
    print("No GRPO checkpoints found. Run training first (section 5).")

## 7. Reset & Iterate

Want to try a different reward function or different parameters?
Run the cell below to clean up, then go back to section 4.

In [ ]:
import shutil, gc

# Free memory
for name in ['model', 'base_model', 'grpo_model']:
    if name in dir():
        exec(f'del {name}')
gc.collect()

# Remove old GRPO checkpoints (keeps SFT model)
shutil.rmtree('checkpoints/grpo-exercise', ignore_errors=True)
print("Cleaned up. Ready for another run.")
print("Go back to section 4 to edit your reward function.")

## Solution (if stuck)

The best reward function uses BFS distance (actual maze distance to exit),
not manhattan distance. It also penalizes loops and gibberish.

In [ ]:
# =================================================================
# SOLUTION: BFS-distance reward function
# =================================================================
# Uncomment and use this if you get stuck. But try designing your
# own first — the learning is in the iteration!
#
# def maze_reward(completion, maze):
#     from src.maze_gen import solve as bfs_solve
#
#     moves = extract_moves(completion)
#     if moves is None:
#         return -1.0
#
#     path = simulate(moves, maze)
#     valid_steps = len(path) - 1
#     optimal_len = len(maze.solution) - 1
#
#     if reached_exit(path, maze):
#         efficiency = optimal_len / max(len(moves), optimal_len)
#         return 0.6 + 0.4 * efficiency
#
#     # BFS distance from final position to exit
#     final_pos = path[-1]
#     remaining = bfs_solve(maze.width, maze.height, maze.walls, final_pos, maze.exit)
#     remaining_dist = len(remaining) - 1 if remaining else optimal_len
#     bfs_progress = max(1.0 - remaining_dist / max(optimal_len, 1), 0.0)
#
#     # Loop penalty
#     unique_cells = len(set(path))
#     loop_penalty = 1.0 - (len(path) - unique_cells) / max(len(path), 1)
#
#     # Combine: BFS progress (70%) + loop penalty (20%) + coverage (10%)
#     coverage = min(valid_steps / max(optimal_len, 1), 1.0)
#     partial = 0.5 * (0.7 * bfs_progress + 0.2 * loop_penalty + 0.1 * coverage)
#     return partial
#
# Key design principles:
#   - BFS distance: measures actual maze distance, not manhattan (ignores walls)
#   - Loop penalty: discourages random wandering that revisits cells
#   - Smooth gradient: every step closer to exit increases reward
#   - Clear gap: solved (0.6+) vs partial (0-0.5) vs gibberish (-1.0)